In [ ]:
# Colab bootstrap: install requirements and stage the course data next to the notebook
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    REPO_URL = "https://github.com/griu/ai-ml-finance-hands-on"
    REPO_DIR = Path("/content/ai-ml-finance-hands-on")
    NOTEBOOK_DIR = Path.cwd()
    DATA_TARGET = NOTEBOOK_DIR / "data"
    REQUIREMENTS_FILE = REPO_DIR / "requirements-colab.txt"

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)],
        check=True,
    )

    shutil.copytree(REPO_DIR / "data", DATA_TARGET, dirs_exist_ok=True)
    print(f"Colab environment detected. Data copied to: {DATA_TARGET.resolve()}")
else:
    print("Local environment detected. Colab bootstrap skipped.")


# 06b — Optional: Temporal, Spatial and Unsupervised Awareness

**Course:** Data Analysis, AI and Machine Learning in Finance
**Type:** Optional self-study extension
**Instructor:** Ferran Carrascosa

---

### What this notebook covers

| Section | Topic |
|---------|-------|
| 0 | Introduction and motivation |
| 1 | Why row independence is not always valid |
| 2 | Temporal awareness — ordering, splits and lag features |
| 3 | Temporal split illustration with Dataset E |
| 4 | Spatial awareness — a conceptual note |
| 5 | Unsupervised awareness — segmentation teaser with Dataset D |
| 6 | Recap and self-practice |

**Datasets:**
- `dataset_E_time_awareness.csv` — 46 annual observations (investment and sales indices)
- `dataset_D_unsupervised_extension.csv` — 10 000 banking customers (numeric features only)

### Why this matters in finance

In the core notebooks (04–06) we worked with a cross-sectional dataset where each
row was **one customer at one point in time**. But many finance problems involve:

- **time order** (stock prices, revenue, macro indicators),
- **spatial or regional context** (branch performance, country risk),
- **no target variable at all** (customer segmentation, anomaly detection).

This notebook builds awareness of these three settings — not deep expertise, just
enough to recognise when the standard supervised workflow needs to be adapted.

---

## Section 1 — Why row independence is not always valid

In Notebooks 04–06 we shuffled rows randomly into train and test sets. That was safe
because each row was a different customer — **the order did not matter**.

But consider these examples:

| Problem | Why order matters |
|---------|-------------------|
| Predict next month's revenue | Past months influence the future — a random split could leak future information into training |
| Compare branch profitability across regions | Nearby branches may share local economic conditions — they are not truly independent |
| Group clients by financial behaviour | There is no target variable — the goal is to discover structure, not predict a label |

> **Key insight:** before choosing a model, ask whether the rows are truly
> exchangeable. If they are ordered in time, located in space, or unlabelled,
> the standard supervised recipe needs adjustments.

---

## Section 2 — Temporal awareness

### Why time order matters

When data has a time dimension, three risks arise if we ignore it:

1. **Look-ahead bias (leakage):** using future information to predict the past.
2. **Autocorrelation:** consecutive observations are similar — random splits break
   this structure and produce overly optimistic metrics.
3. **Distribution shift:** the relationship between features and target may change
   over time.

### Chronological split vs random split

| Split type | How it works | When to use |
|-----------|-------------|-------------|
| Random | Shuffle all rows, pick 80 % train / 20 % test | Cross-sectional data (one row = one independent entity) |
| Chronological | Train on earlier periods, test on later periods | Time-ordered data (forecast, macro indicators, price series) |

### Lag features and rolling means

Two common techniques to **encode temporal context** into features:

- **Lag feature:** the value of a variable *n* periods ago.
  `sales_lag1 = sales shifted by 1 period`
- **Rolling mean:** the average of the last *n* periods.
  `sales_roll3 = mean of the last 3 periods`

These features let a model "see" recent history without looking into the future.

---

## Section 3 — Temporal split illustration with Dataset E

Dataset E contains **46 annual observations** of two macro-style indices (investment
and sales). We will use it to illustrate chronological ordering, lag features,
rolling means and a proper temporal train/test split.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Resolve data directory — works from the project root or notebooks/
_candidates = [Path("data/processed"), Path("../data/processed")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/. "
        "Launch the notebook from the project root or the notebooks/ folder."
    )

df_e = pd.read_csv(DATA_DIR / "dataset_E_time_awareness.csv", parse_dates=["date"])
df_e = df_e.sort_values("date").reset_index(drop=True)

print(f"Dataset E: {df_e.shape[0]} rows × {df_e.shape[1]} columns")
print(f"Period: {df_e['year'].min()} – {df_e['year'].max()}")
df_e.head(10)

### Creating lag and rolling features

In [ ]:
# Lag features — value from the previous year
df_e["investment_lag1"] = df_e["investment_index"].shift(1)
df_e["sales_lag1"] = df_e["sales_index"].shift(1)

# Rolling means — average of the last 3 years
df_e["investment_roll3"] = df_e["investment_index"].rolling(3).mean()
df_e["sales_roll3"] = df_e["sales_index"].rolling(3).mean()

df_e.head(8)

> Notice the `NaN` values in the first rows: lag and rolling features require
> past data that does not exist for the earliest observations. This is normal —
> we drop those rows before modelling.

### Chronological train / test split

We split by time: everything **before a cut-off year** goes to training;
everything **from that year onward** goes to testing.

In [ ]:
CUT_YEAR = 2015

train = df_e[df_e["year"] < CUT_YEAR].dropna().copy()
test  = df_e[df_e["year"] >= CUT_YEAR].dropna().copy()

print(f"Train: {len(train)} rows  ({train['year'].min()}–{train['year'].max()})")
print(f"Test:  {len(test)} rows  ({test['year'].min()}–{test['year'].max()})")

### Visualising the temporal split

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(train["year"], train["sales_index"], "o-", label="Train", color="steelblue")
ax.plot(test["year"], test["sales_index"], "s--", label="Test", color="darkorange")
ax.axvline(CUT_YEAR, color="grey", linestyle=":", linewidth=1, label=f"Cut-off ({CUT_YEAR})")
ax.set_xlabel("Year")
ax.set_ylabel("Sales index")
ax.set_title("Chronological train / test split — Dataset E")
ax.legend()
plt.tight_layout()
plt.show()

> **Why not a random split here?** If we shuffled the years randomly, the
> training set could include 2020 while the test set includes 2010. The model
> would effectively "know the future" — producing unrealistically good metrics
> that would not hold in a real forecast.

### Why a random split would be misleading

In [ ]:
# Demonstrate the problem: random split mixes time periods
np.random.seed(42)
random_idx = np.random.permutation(len(df_e.dropna()))
rand_train = df_e.dropna().iloc[random_idx[:30]]
rand_test  = df_e.dropna().iloc[random_idx[30:]]

print("Random-split TRAIN years (sample):", sorted(rand_train["year"].values[:10]))
print("Random-split TEST  years (sample):", sorted(rand_test["year"].values[:5]))
print()
print("Notice how past and future are mixed — this would cause leakage in a real forecast.")

---

## Section 4 — Spatial awareness (a conceptual note)

### When location matters

Some finance and business problems have a **geographic or regional dimension**:

| Example | Why spatial context matters |
|---------|---------------------------|
| Branch profitability | Nearby branches share local economic conditions |
| Real-estate valuation | Location is the single strongest predictor |
| Country risk modelling | Neighbouring economies may be correlated |
| Insurance claims | Weather and infrastructure vary by region |

### Key ideas

- **Spatial dependence:** units that are close together tend to behave more similarly
  than units that are far apart.
- **Region as a feature:** even without coordinates, a region or country variable
  captures some spatial structure (we used `geography` in NB04–06).
- **Spatial validation:** in some problems, training on one region and testing on
  another is more realistic than a random split.

> **For this course** we will not implement spatial models or use mapping libraries.
> The important takeaway is: **if your data has a geographic dimension, think about
> whether nearby observations are truly independent before applying a random split.**

---

## Section 5 — Unsupervised awareness: a segmentation teaser

### When there is no target variable

All models in NB04–06 were **supervised**: we had a known target (`attrition_flag`)
and trained the model to predict it. But sometimes:

- the business question is "**what groups exist?**" rather than "predict this label",
- the goal is to **discover structure** that humans have not defined in advance,
- the dataset does not contain a target column at all.

This is the domain of **unsupervised learning**: clustering, dimensionality reduction,
anomaly detection.

### When segmentation is useful in finance

| Use case | What unsupervised learning provides |
|----------|-------------------------------------|
| Customer segmentation | Groups of clients with similar financial behaviour — useful for targeted marketing or risk tiers |
| Portfolio clustering | Groups of assets with correlated returns — useful for diversification analysis |
| Anomaly detection | Transactions that do not fit any known pattern — useful for fraud screening |
| Feature compression | Many correlated variables reduced to a smaller set — useful when the number of features is very large |

### Quick preview with Dataset D

Dataset D contains **10 000 banking customers** with 8 numeric features — the same
features used in the core notebooks, but **without the target column**. This makes
it a natural fit for unsupervised exploration.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load Dataset D (no target variable — unsupervised)
df_d = pd.read_csv(DATA_DIR / "dataset_D_unsupervised_extension.csv")
print(f"Dataset D: {df_d.shape[0]:,} rows × {df_d.shape[1]} columns")
print(f"Columns: {list(df_d.columns)}")
df_d.head()

In [ ]:
# Standardise before clustering (distance-based methods need scaling)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_d)

# K-Means with 3 clusters (a simple starting choice)
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df_d["cluster"] = km.fit_predict(X_scaled)

print("Cluster sizes:")
print(df_d["cluster"].value_counts().sort_index())

> We chose 3 clusters as a simple illustration. In practice, you would
> experiment with different values and use techniques like the elbow method
> (covered in Notebook 06d) to guide the choice.

In [ ]:
# Cluster profiles — mean of each feature per cluster
profile = df_d.groupby("cluster").mean().round(1)
profile

### Interpreting the clusters

Look at the table above and try to **give each cluster a name** based on its
financial profile. For example:

- A cluster with high balance, high salary and low complaints might be
  "premium low-risk clients".
- A cluster with many products and high digital usage might be "digitally
  active multi-product clients".

> **Important:** cluster labels are not ground truth — they are descriptions we
> assign after the fact based on the data patterns. The algorithm groups similar
> rows; the **business interpretation** is our job.

In [ ]:
# Quick 2D visualisation using the first two scaled features
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X_scaled[:, 0], X_scaled[:, 1],
                     c=df_d["cluster"], cmap="Set2", alpha=0.4, s=10)
ax.set_xlabel("Feature 1 (credit_score, scaled)")
ax.set_ylabel("Feature 2 (age, scaled)")
ax.set_title("K-Means clusters — 2D projection (Dataset D)")
plt.colorbar(scatter, ax=ax, label="Cluster")
plt.tight_layout()
plt.show()

> This 2D view uses only two features. The actual clustering operates in all
> 8 dimensions — the plot is an approximation, not the full picture.
> Notebook 06d will show PCA as a better way to reduce dimensions for
> visualisation.

---

## Section 6 — Recap

### What to remember

| Concept | Key takeaway |
|---------|-------------|
| **Temporal structure** | When data has a time dimension, use a chronological split — never shuffle future into training |
| **Lag & rolling features** | Encode recent history as features; accept that early rows will have missing values |
| **Spatial dependence** | Nearby entities may share patterns — consider region-based splits or region as a feature |
| **Unsupervised learning** | When there is no target, use clustering or dimensionality reduction to discover structure |
| **Interpretation** | Clusters and components need business interpretation — the algorithm finds patterns, not answers |

### Bridge to other notebooks

- **Notebook 06d** goes deeper into K-Means, PCA and adds a sentiment example.
- The **core modelling workflow** (NB04–06) remains the foundation — this notebook
  simply widens your awareness of settings where that workflow needs adaptation.

---

### Self-practice

1. In the temporal split above, change the cut-off year to 2010. How does the
   train/test balance change? Is that a problem?
2. Create a lag-2 feature (`investment_lag2`) for Dataset E. Why might a
   longer lag sometimes be useful?
3. Re-run K-Means on Dataset D with 5 clusters. Do the cluster profiles
   tell a different story?
4. Write one sentence explaining why a random train/test split would be
   dangerous for predicting next quarter's stock returns.
5. Name one finance use case where unsupervised learning would be more
   appropriate than supervised prediction.

---

*End of Notebook 06b — Optional: Temporal, Spatial and Unsupervised Awareness.*